# Documentación del Proceso de Transformación – Capa Silver  
## Arquitectura Medallion en Databricks

## 1. Introducción

El presente notebook implementa el proceso de **transformación de datos** correspondiente a la **capa Silver** dentro de una arquitectura **Medallion**, utilizando **Apache Spark**, **Delta Lake** y operaciones de tipo **UPSERT (MERGE)**.

La capa Silver tiene como objetivo principal **refinar los datos provenientes de la capa Bronze**, aplicando tareas de:
- Limpieza
- Normalización
- Enriquecimiento
- Deduplicación
- Auditoría

El resultado son tablas confiables y consistentes, listas para ser utilizadas en análisis avanzados o en la construcción de la capa Gold.

---

## 2. Rol de la Capa Silver en la Arquitectura Medallion

Dentro de la arquitectura Medallion, la capa Silver cumple las siguientes funciones:

- Consumir datos crudos desde la capa Bronze
- Aplicar reglas de calidad de datos
- Resolver duplicados y registros obsoletos
- Incorporar columnas derivadas y de auditoría
- Mantener las tablas actualizadas de forma incremental

A diferencia de la capa Bronze, los datos en Silver ya poseen un **significado más cercano al negocio**, aunque todavía no están agregados ni modelados para consumo final.

---

## 3. Descripción General del Proceso

El proceso de transformación Silver sigue un patrón común para todas las entidades (`customers`, `drivers`, `locations`, `payments`, `vehicles`) y se compone de las siguientes etapas:

1. Lectura de tablas Bronze
2. Limpieza y transformación de atributos
3. Deduplicación basada en claves lógicas
4. Incorporación de columnas de auditoría
5. Carga incremental mediante UPSERT en tablas Silver

---

### 3.1 Lectura de Datos desde la Capa Bronze

Cada entidad es leída directamente desde su correspondiente tabla Delta en la capa Bronze:

Catalog → pysparkdbt → bronze → Tables → {entity}

Estas tablas contienen los datos crudos ya persistidos en formato **Delta Lake**, pero aún **sin validaciones ni estandarizaciones**, por lo que pueden presentar inconsistencias propias del origen de los datos.

---

### 3.2 Transformaciones y Limppieza de Datos

Durante esta etapa se aplican **transformaciones específicas según cada entidad**, con el objetivo de mejorar la calidad, consistencia y utilidad de los datos. Entre las principales transformaciones realizadas se destacan las siguientes.

### 3.2.1 Customers y Drivers

Para las entidades **customers** y **drivers** se aplican las siguientes transformaciones:

- Normalización del número telefónico mediante la eliminación de caracteres no numéricos.
- Construcción de la columna `full_name` a partir de la concatenación de `first_name` y `last_name`.
- Eliminación de columnas redundantes o innecesarias para el análisis.
- Extracción del dominio del correo electrónico (`domain`) en el caso de la entidad **customers**.

Estas transformaciones permiten **estandarizar los datos** y facilitan su utilización en procesos analíticos posteriores.

---

### 3.2.2 Payments

En la entidad **payments** se genera una nueva columna derivada denominada `online_payment_status`, la cual combina la información proveniente de:

- El método de pago.
- El estado del pago.

Esta transformación permite clasificar los pagos en **categorías funcionales**, facilitando análisis financieros y de comportamiento.

---

### 3.2.3 Vehicles

Para la entidad **vehicles**, se normaliza la columna `make`, convirtiendo su contenido a **mayúsculas**, con el objetivo de asegurar la consistencia de los valores categóricos y evitar duplicidades semánticas.

---

## 4. Deduplicación de Registros

Para todas las entidades se aplica un proceso de **deduplicación**, utilizando:

- Una clave lógica representativa de la entidad (por ejemplo: `customer_id`, `driver_id`, `payment_id`, etc.).
- La columna `last_updated_timestamp` como criterio de ordenamiento temporal.

El proceso conserva únicamente el **registro más reciente** para cada clave lógica, descartando versiones anteriores del mismo identificador.

Este paso es fundamental para garantizar la **unicidad** y **actualidad** de los datos en la capa Silver.

---

## 5. Auditoría del Proceso

Se incorpora la columna de auditoría `process_timestamp`, la cual registra el momento exacto en que cada registro fue procesado por el pipeline.

Esta columna permite:

- Trazabilidad del procesamiento de datos.
- Auditoría del flujo de información.
- Análisis temporal del funcionamiento del pipeline.

### Ejemplo conceptual

| customer_id | signup_date | process_timestamp |
|------------|------------|-------------------|
| 10 | 2022-05-01 | 2025-01-10 14:32 |

---

## 6. Carga Incremental en la Capa Silver (UPSERT)

Antes de persistir los datos, el proceso verifica la existencia de la tabla correspondiente en la capa Silver:

- Si la tabla **no existe**, se crea una nueva tabla Delta y se cargan los datos iniciales.
- Si la tabla **ya existe**, se ejecuta una operación de **MERGE (UPSERT)**.

La operación UPSERT permite:

- Insertar nuevos registros.
- Actualizar registros existentes cuando se dispone de una versión más reciente.

Este enfoque asegura que las tablas Silver se mantengan **sincronizadas** con la información más actual proveniente de la capa Bronze.

---

## 7. Persistencia de Datos en la Capa Silver

Los datos transformados se almacenan en la siguiente ubicación del catálogo:

Catalog → pysparkdbt → silver → Tables → {entity}

Estas tablas representan la versión **depurada y consolidada** de los datos, y constituyen la base para:

- Modelos analíticos.
- Agregaciones.
- Indicadores de negocio correspondientes a la capa Gold.

---

## 10. Conclusión

El proceso implementado para la capa Silver permite transformar los datos crudos provenientes de la capa Bronze en conjuntos de datos **confiables, normalizados y auditables**.

El uso de procesos de deduplicación, columnas de auditoría y cargas incrementales mediante operaciones **UPSERT** garantiza:

- Calidad de datos.
- Consistencia temporal.
- Escalabilidad del pipeline.

Este diseño se alinea con **prácticas profesionales de ingeniería de datos** y facilita la construcción de capas analíticas posteriores dentro de una arquitectura **Medallion**.


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from typing import List
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from pyspark.sql.functions import col, when


In [0]:
import os
import sys

In [0]:
current_dir = os.getcwd()
sys.path.append(current_dir)


### **CUSTOMERS**

In [0]:
df_cust = spark.read.table("pysparkdbt.bronze.customers")

In [0]:
#display(df_cust)

In [0]:
# Crea la columna domain a partir de la columna email, separando el contenido por el carácter @ y generando un array con las partes del correo
df_cust = df_cust.withColumn("domain",split(col('email'),'@'))
#display(df_cust.select("email","domain"))

In [0]:
# Elimina todos los caracteres no numericos de la columna "phone_number"
df_cust = df_cust.withColumn("phone_number",regexp_replace("phone_number",r"[^0-9]",""))
#display(df_cust.select("phone_number"))

In [0]:
# Genera la columna full_name concatenando first_name y last_name con un espacio como separador, elimina las columnas originales
df_cust = df_cust.withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name")))
df_cust = df_cust.drop("first_name","last_name")
#display(df_cust.select("full_name"))


In [0]:
from utils.custom_utils import transformations


REPASO TEORICO
A- PIPELINE

 la funcion "process_timestamp": Agrega una columna de auditoría con la marca temporal del procesamiento. La función incorpora la columna process_timestamp, que registra el momento exacto en el que el DataFrame es procesado dentro del pipeline.

 Un pipeline de datos es: La secuencia de pasos por la que pasan los datos desde que se leen hasta que se guardan para su uso final.
 Materia prima → limpieza → armado → control → producto final
 Datos crudos → transformación → validación → carga → consumo

 Para este caso:  
Bronze (raw)  
   │  
   ├─ lectura de tabla  
   │  
   ├─ limpieza (phone_number, email, full_name)  
   │  
   ├─ deduplicación (dedup)  
   │  
   ├─ auditoría (process_timestamp)  
   │  
   └─ carga en Silver (upsert)  

   df = df.withColumn("process_timestamp", current_timestamp()) --> Marca cuándo pasó ese DataFrame por este punto del pipeline.  

| customer_id | signup_date | process_timestamp |
| ----------- | ----------- | ----------------- |
| 10          | 2022-05-01  | 2025-01-10 14:32  |


B- MARGE
MERGE es una operación que permite sincronizar una tabla con un DataFrame combinando INSERT y UPDATE en un solo paso.


In [0]:
# Se genera un nuevo DataFrame (cust_df_trns) a partir del original, en el cual se eliminan los registros duplicados según la clave lógica customer_id.
# Para cada customer_id, los registros se ordenan por la columna last_updated_timestamp en orden descendente, conservando únicamente el registro más reciente y descartando los más antiguos.


cust_obj = transformations()

cust_df_trns = cust_obj.dedup(df_cust, ['customer_id'], 'last_updated_timestamp', order='desc')
#display(df_cust.select("customer_id"))

In [0]:
# Agrega una columna de auditoría con la marca temporal del procesamiento
df_cust = cust_obj.process_timestamp(cust_df_trns)

Base de datos en SPARK

¿Existe silver.customers?  
│  
├── NO  
│   └── Crear tabla Delta y cargar datos iniciales  
│  
└── SÍ  
    └── Ejecutar MERGE (UPSERT) incremental (actualiza la informacion) 

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.customers"):
    df_cust.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.customers")
else:
    cust_obj.upsert(spark,df_cust,['customer_id'],'customers','last_updated_timestamp')


In [0]:
# %sql
# SELECT COUNT(*) FROM PYSPARKDBT.SILVER.customers

### **DRIVERS**

In [0]:
df_driver = spark.read.table("pysparkdbt.bronze.drivers")
# display(df_driver)

In [0]:
# Elimina todos los caracteres no numericos de la columna "phone_number"
df_driver = df_driver.withColumn("phone_number",regexp_replace("phone_number",r"[^0-9]",""))
#display(df_driver.select("phone_number"))

# Genera la columna full_name concatenando first_name y last_name con un espacio como separador, elimina las columnas originales
df_driver = df_driver.withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name")))
df_driver = df_driver.drop("first_name","last_name")
#display(df_cust.select("full_name"))


In [0]:
driver_obj = transformations()

In [0]:
df_driver = driver_obj.dedup(df_driver, ['driver_id'], 'last_updated_timestamp', order='desc')
df_driver = driver_obj.process_timestamp(df_driver)

if not spark.catalog.tableExists("pysparkdbt.silver.drivers"):
    df_driver.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.drivers")
else:
    driver_obj.upsert(spark,df_cust,['driver_id'],'drivers','last_updated_timestamp')


In [0]:
%sql
SELECT COUNT(*) FROM PYSPARKDBT.SILVER.drivers

### **LOCATIONS**

In [0]:
df_loc = spark.read.table("pysparkdbt.bronze.locations")
# display(df_loc)

In [0]:
loc_obj = transformations()

In [0]:
df_loc = loc_obj.dedup(df_loc, ['location_id'], 'last_updated_timestamp', order='desc')
df_loc = loc_obj.process_timestamp(df_loc)

if not spark.catalog.tableExists("pysparkdbt.silver.locations"):
    df_loc.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.locations")
else:
    loc_obj.upsert(spark,df_loc,['location_id'],'locations','last_updated_timestamp')

In [0]:
%sql
SELECT COUNT(*) FROM PYSPARKDBT.SILVER.locations

### PAYMENTS

In [0]:
df_pay = spark.read.table("pysparkdbt.bronze.payments")
# display(df_pay)

In [0]:
df_pay = df_pay.withColumn("online_payment_status",
            when( ((col('payment_method')=='Card') & (col('payment_status')=='Success')), "Online-success")
            .when( ((col('payment_method')=='Card') & (col('payment_status')=='Failed')), "Online-failed")
            .when( ((col('payment_method')=='Card') & (col('payment_status')=='Pending')), "Online-pending")
            .otherwise("offline")
            )

# display(df_pay)                          

In [0]:
pay_obj = transformations()

In [0]:
df_pay = pay_obj.dedup(df_pay, ['payment_id'], 'last_updated_timestamp', order='desc')
df_pay = pay_obj.process_timestamp(df_pay)

if not spark.catalog.tableExists("pysparkdbt.silver.payments"):
    df_pay.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.payments")
else:
    pay_obj.upsert(spark,df_pay,['payment_id'],'payments','last_updated_timestamp')

In [0]:
%sql
SELECT COUNT(*) FROM PYSPARKDBT.SILVER.payments

### VEHICLES

In [0]:
df_veh = spark.read.table("pysparkdbt.bronze.vehicles")
# display(df_veh)

In [0]:
df_veh = df_veh.withColumn("make", upper(col("make")))
veh_obj = transformations()
df_veh = veh_obj.dedup(df_veh, ['vehicle_id'], 'last_updated_timestamp', order='desc')
df_veh = veh_obj.process_timestamp(df_veh)

if not spark.catalog.tableExists("pysparkdbt.silver.vehicles"):
    df_veh.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.vehicles")
else:
    veh_obj.upsert(spark,df_veh,['vehicle_id'],'vehicles','last_updated_timestamp')



In [0]:
%sql
SELECT COUNT(*) FROM PYSPARKDBT.SILVER.vehicles